# 太湖数据工厂 SIM-V1 独立质量审计

## tl;dr

当前 `mvp_meiliangwan_2024` 可以运行并生成结构化 SIM-V1 数据，35 个定向测试通过、发布文件哈希无错误；但独立审计发现 4 个 Critical、8 个 High、2 个 Medium 问题，因此不能进入正式实验。最严重的问题是全湖记录其实复制梅梁湾、部分校准分支读取训练截止日之后的数据、仿真卫星观测被重标为非合成观测标签，以及水位 100% 落在硬下界。

## Context & Methods

本笔记本只读取现有运行目录和发布包，不重跑或修改生成器。审计粒度包括：冻结网格、逐日网格潜在状态、任务标签、训练样本、观测层、校准参数、切分和发布包。

### Key Assumptions

- 正式实验目标以用户提供的 V1.0 清单为准；当前输出按 MVP 单年梅梁湾范围评估。
- `is_synthetic`、`is_ground_truth` 和 `label_status` 必须从上游证据不可逆传递。
- 时间切分要求所有生成器校准输入不晚于 2024-08-28。

## Data

读取已落盘的 SIM-V1 运行产物，并调用同目录的独立审计脚本。

In [1]:
from pathlib import Path
import sys
import pandas as pd

audit_dir = Path.cwd()
if not (audit_dir / 'audit_data_generation.py').exists():
    audit_dir = Path(r'D:/Project/fuwai/项目完整汇总_2026-08-31/01_我们的开发/reports/data-generation-audit-2026-09-04')
sys.path.insert(0, str(audit_dir))
from audit_data_generation import run_audit

metrics, findings, details, extra = run_audit()
extra['summary']

{'status': 'NOT_READY_FOR_FORMAL_EXPERIMENT',
 'mvp_status': 'STRUCTURALLY_RUNNABLE_WITH_CRITICAL_DATA_QUALITY_DEFECTS',
 'critical': 4,
 'high': 8,
 'medium': 2,
 'tests_observed': '35 targeted tests passed; independent audit identifies gaps not covered by those tests',
 'scope': 'mvp_meiliangwan_2024 / baseline / seed 20260904 / SIM-V1',
 'hash_failures': [],
 'missing_release_contract_files': ['lineage/source_registry.csv',
  'generation/parameter_sets.parquet',
  'lineage/transformation_log.jsonl',
  'quality/leakage_audit.csv',
  'data/target_observation_daily.parquet',
  'data/dynamic_features_grid_daily.parquet']}

## Results

### 1. 分级问题

下表是独立复算后的问题，不采信内部质量报告的总 PASS 作为结论。

In [2]:
severity_order = pd.CategoricalDtype(['Critical', 'High', 'Medium', 'Low'], ordered=True)
findings_view = findings.copy()
findings_view['severity'] = findings_view['severity'].astype(severity_order)
findings_view.sort_values(['severity', 'issue_id'])[['issue_id', 'severity', 'finding', 'evidence']]

,issue_id,severity,finding,evidence
0,DG-001,Critical,TAIHU_WHOLE is not a full-lake simulation in t...,All 366 daily whole-lake rows equal TAIHU_ML; ...
1,DG-002,Critical,The calibration cutoff is not enforced in seve...,"After 2024-08-28, history contains 15 ground-t..."
2,DG-003,Critical,Synthetic satellite retrievals are relabeled a...,"The observation table has 36,448 is_synthetic=..."
4,DG-005,Critical,Physical clipping hides model instability inst...,Water level is at a hard bound for 100.0% of d...
3,DG-004,High,Training features come from the omniscient lat...,"Station observation output has 0 rows, but sam..."
5,DG-006,High,Chlorophyll-a seasonality is too weak for the ...,"Mean Chl-a is 4.037 ug/L, CV=0.081, and Jun-Au..."
6,DG-007,High,Chronological splits contain single-class bina...,6 task/spatial/split groups are 0% or 100% pos...
7,DG-008,High,The MVP does not cover all seven tasks at 1 km...,"Grid labels exist for ['T1', 'T2', 'T5']; miss..."
8,DG-009,High,The release manifest's missing=[] does not tes...,Missing audited contract artifacts: lineage/so...
10,DG-011,High,The overall quality verdict is PASS even with ...,A13 is warning while quality_summary says PASS...


### 2. 关键量化证据

In [3]:
wanted = [
    'simulation_area_share', 'whole_lake_equals_meiliang_bay',
    'ground_truth_wq_rows_after_fit_cutoff', 'water_level_rows_after_fit_cutoff',
    'synthetic_satellite_observation_rows', 'satellite_labels_marked_non_synthetic',
    'station_observation_rows', 'water_level_at_bounds',
    'chlorophyll_a_summer_vs_winter', 'degenerate_binary_task_split_groups',
    'formal_contract_files_missing', 'row_lineage_rows',
    'mee_rows_with_no_year_in_timestamp', 'release_hash_failures'
]
metrics[metrics['metric'].isin(wanted)].set_index('metric').loc[wanted].reset_index()

,metric,value,unit,interpretation
0,simulation_area_share,0.208298,ratio,MVP area divided by frozen full-lake area
1,whole_lake_equals_meiliang_bay,1.000000,boolean,1 means whole-lake rows duplicate the only sim...
2,ground_truth_wq_rows_after_fit_cutoff,15.000000,rows,Rows accessible to unfiltered nutrient/algae fit
3,water_level_rows_after_fit_cutoff,3.000000,rows,Rows accessible to unfiltered hydrology fit
4,synthetic_satellite_observation_rows,36448.000000,rows,Generated from latent simulation state
5,satellite_labels_marked_non_synthetic,70.000000,rows,Contradicts their upstream observation origin
6,station_observation_rows,0.000000,rows,Actual station observation layer used for MVP
7,water_level_at_bounds,1.000000,ratio,Hydrology values exactly at 2 m or 5 m
8,chlorophyll_a_summer_vs_winter,0.008145,ratio,Jun-Aug mean divided by Dec-Feb mean minus one
9,degenerate_binary_task_split_groups,6.000000,groups,Unique-date groups with only one class


### 3. 边界钳位和二分类切分

In [4]:
clipping = details[details['table'].eq('clipping')][['variable', 'rows', 'at_lower_bound', 'at_upper_bound', 'bound_share']]
date_balance = details[details['table'].eq('date_balance')][['target_metric', 'spatial_type', 'split', 'samples', 'positives', 'positive_rate']]
display(clipping)
display(date_balance)

,variable,rows,at_lower_bound,at_upper_bound,bound_share
0,total_phosphorus,196176.0,1.0,17.0,0.000092
1,total_nitrogen,196176.0,0.0,74750.0,0.381035
2,ammonia_nitrogen,196176.0,0.0,1.0,0.000005
3,dissolved_oxygen,196176.0,169.0,0.0,0.000861
4,water_temperature,196176.0,925.0,0.0,0.004715


,target_metric,spatial_type,split,samples,positives,positive_rate
5,T1,grid,test,11792.0,2371.0,0.201069
6,T1,grid,train,71824.0,27325.0,0.380444
7,T1,grid,validation,7504.0,7504.0,1.000000
8,T1,lake,test,40.0,22.0,0.550000
9,T1,lake,train,240.0,123.0,0.512500
10,T1,lake,validation,25.0,25.0,1.000000
11,T1,zone,test,40.0,40.0,1.000000
12,T1,zone,train,240.0,237.0,0.987500
13,T1,zone,validation,25.0,25.0,1.000000
14,T7,lake,test,40.0,27.0,0.675000


## Takeaways

1. 当前产物适合作为工程原型，不适合作为正式算法训练或效果证明数据。
2. 必须先修复训练截止日过滤、仿真身份传递、全湖聚合和水位参数错误，再重新生成。
3. 训练特征必须来自可见观测/预报层，而不是直接读取潜在真值。
4. 重新运行至少五年全湖数据后，再按真实日期事件检查每个 split 的正负样本。
5. 发布门禁应分为结构完整性、仿真可信度和正式训练就绪度，不能只给一个总 PASS。